# 02. Exploratory Data Analysis

> Get a feel for the dataset before any processing: per-channel signal statistics, label timeline, gesture trial counts, and finger-flexion traces from the data glove.

The goal is to surface anomalies (bad channels, drifts, class imbalance, glove dropouts) that will inform the preprocessing and epoching choices in later notebooks. No code is exported from here — it is purely investigative. Each plot is also written to `outputs/02_eda/` so the artifacts survive even after `nbdev-clean` strips inline cell outputs.

## Setup

In [ ]:
%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
from br41n_ecog_hand_pose.data import load_ecog, FS, GESTURE_NAMES, FINGER_NAMES

plt.rcParams.update({
    'axes.grid':      True,
    'grid.linestyle': ':',
    'grid.linewidth': 0.5,
    'grid.alpha':     0.6,
})

def _project_root():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'pyproject.toml').exists(): return parent
    return Path.cwd()

OUTPUT_DIR = _project_root() / 'outputs' / '02_eda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load the recording

Load the full session once and reuse the resulting `ECoGRecording` for every plot below.

In [ ]:
#| eval: false
rec = load_ecog()
print(f'duration:    {rec.duration_s:.1f} s  ({rec.n_samples:,} samples @ {rec.fs} Hz)')
print(f'ecog shape:  {rec.ecog.shape}')
print(f'glove shape: {rec.glove.shape}')

## Label timeline

Visualize CH62 across the full recording. Each non-zero plateau is a 2 s gesture cue; the gaps between are the 2–3 s rest screens. A clean rectangular pattern means cue detection in `04_epoching` will be trivial.

In [ ]:
#| eval: false
fig, ax = plt.subplots(figsize=(12, 2))
ax.plot(rec.time, rec.labels, lw=0.5)
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels([GESTURE_NAMES[i] for i in [0, 1, 2, 3]])
ax.set_xlabel('time (s)')
ax.set_title('CH62 paradigm labels — full recording')
fig.savefig(OUTPUT_DIR / '01_label_timeline.png', dpi=250, bbox_inches='tight')
plt.tight_layout(); plt.show()

![Label timeline](../outputs/02_eda/01_label_timeline.png)

**Observations.** The label channel is a textbook rectangular pulse train across the full ~7 minute recording. Plateaus are well-separated and uniform in duration, so detecting cue onsets via `0 → non-zero` transitions in `04_epoching` is straightforward — no debouncing or smoothing needed. Class assignment per cue is also unambiguous: each plateau holds a single integer code throughout its duration.

## Trial counts per class

Cue onsets are samples where the label transitions `0 → {1, 2, 3}`. Counting by class verifies the documented 90-trial paradigm and surfaces any class imbalance.

In [ ]:
#| eval: false
labels = rec.labels
onsets = np.where((labels[1:] != 0) & (labels[:-1] == 0))[0] + 1
classes_at_onset = labels[onsets]
uniq, counts = np.unique(classes_at_onset, return_counts=True)
for c, n in zip(uniq, counts):
    print(f'{GESTURE_NAMES[c]:>5}: {n} trials')
print(f'total: {len(onsets)} trials')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar([GESTURE_NAMES[c] for c in uniq], counts)
ax.set_ylabel('trial count')
ax.set_title('Trials per gesture class')
fig.savefig(OUTPUT_DIR / '02_trial_counts.png', dpi=250, bbox_inches='tight')
plt.tight_layout(); plt.show()

![Trial counts per class](../outputs/02_eda/02_trial_counts.png)

**Observations.** Perfectly balanced: 30 trials per class, 90 total. No class re-weighting needed in any classifier, and stratified 5-fold CV gives exactly 6 trials per class per test fold. The `relax` code (0) doesn't appear here because we count *cues*, not samples — the 0 state is the inter-trial rest screen, used as baseline rather than as a fourth class.

## Per-channel signal stats

Per-channel mean and std flag dead channels (std ≈ 0), saturated channels (std orders of magnitude above the rest), and DC drift (large mean offsets — expected here since the data is DC-coupled, but extreme values still indicate a problem).

In [ ]:
#| eval: false
mean = rec.ecog.mean(axis=1)
std  = rec.ecog.std(axis=1)
fig, axs = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axs[0].bar(np.arange(60), mean); axs[0].set_ylabel('mean (\u03bcV)')
axs[1].bar(np.arange(60), std);  axs[1].set_ylabel('std (\u03bcV)')
axs[1].set_xlabel('channel index (0 = CH2)')
fig.suptitle('Per-channel ECoG mean and std')
fig.savefig(OUTPUT_DIR / '03_channel_stats.png', dpi=250, bbox_inches='tight')
plt.tight_layout(); plt.show()

![Per-channel mean and std](../outputs/02_eda/03_channel_stats.png)

**Observations.** Mean values span roughly `−110 000` to `+75 000` μV — these are DC offsets from the DC-coupled recording, not signal. The bandpass in `03_preprocessing` eliminates them entirely. Per-channel std varies ~10× across the 60-channel grid (≈ 600 → 5 800 μV); this is the natural amplitude variation across motor/sensory cortex, not a quality issue, since no channel collapses to ≈ 0 (which would indicate a dead electrode). The MAD-based `find_bad_channels` in `03` will pick up any outliers in the post-bandpass signal.

## Spectral overview (PSD)

Welch PSD per channel, overlaid. Two things to look for: line-noise peaks (50 or 60 Hz plus harmonics — informs the notch filter in `03_preprocessing`), and channels whose spectrum is very different from the rest (candidates for rejection).

In [ ]:
#| eval: false
freqs, psd = welch(rec.ecog, fs=rec.fs, nperseg=2 * rec.fs)
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(freqs, psd.T, lw=0.4, alpha=0.5)
ax.set_xlim(0, 200)
ax.set_xlabel('frequency (Hz)')
ax.set_ylabel('PSD (\u03bcV\u00b2/Hz)')
ax.set_title('Welch PSD per channel — line-noise peaks and channel outliers')
fig.savefig(OUTPUT_DIR / '04_psd.png', dpi=250, bbox_inches='tight')
plt.tight_layout(); plt.show()

![Welch PSD per channel](../outputs/02_eda/04_psd.png)

**Observations.**
- **Strong 50 Hz line-noise peaks** plus harmonics at 100 and 150 Hz — confirms a European recording site. The notch filter in `03_preprocessing` is configured `freq=50, harmonics=3`, which removes all three peaks in one pass.
- **The high-gamma band (70–170 Hz)** — the workhorse of ECoG hand decoding — is otherwise clean and follows a smooth 1/f decay between the line-noise notches. This is exactly what we want.
- A small fraction of channels deviate from the bulk envelope (typically grid-edge channels picking up environmental noise). They didn't trigger the bad-channel threshold, but worth revisiting if downstream accuracy plateaus.

## Glove traces per class

Average finger flexion in a 5 s window after each cue onset, per gesture class. The three gestures should produce visibly distinct finger-flexion patterns (e.g. all fingers flexed for `fist`, only index+middle extended for `peace`). If the patterns aren't separable here, no decoder is going to help.

In [ ]:
#| eval: false
window_s = 5
n_win = int(window_s * rec.fs)
fig, axs = plt.subplots(3, 1, figsize=(8, 9), sharey=True, constrained_layout=True)
for ax, c in zip(axs, [1, 2, 3]):
    trials = onsets[classes_at_onset == c]
    trials = trials[trials + n_win <= rec.glove.shape[1]]
    snippets = np.stack([rec.glove[:, o:o + n_win] for o in trials])
    mean_trace = snippets.mean(axis=0)
    t_win = np.arange(n_win) / rec.fs
    for fi, fname in enumerate(FINGER_NAMES):
        ax.plot(t_win, mean_trace[fi], label=fname)
    ax.set_title(f'{GESTURE_NAMES[c]} (n={len(trials)})')
    ax.set_xlabel('time from cue onset (s)')
    ax.set_ylabel('flexion')
    ax.legend(fontsize=8, loc='best')
fig.suptitle('Mean data-glove traces per gesture class')
fig.savefig(OUTPUT_DIR / '05_glove_traces.png', dpi=250, bbox_inches='tight')
plt.show()

![Mean glove traces per gesture class](../outputs/02_eda/05_glove_traces.png)

**Observations.**
- **Reaction time ~400–500 ms.** Movement starts well after the cue, not at `t=0`. For *movement* decoding, tighten the epoch window to e.g. `tmin=0.5, tmax=2.5`. For *cue/intent* decoding (the classification task) keep `tmin=0` so the early premotor ECoG response is included.
- **`open` is kinematically flat.** The hand barely moves above baseline during the open cue — its glove signal is indistinguishable from rest. Consequence for `08_finger_regression`: the open trials contribute almost zero target variance, so per-finger Pearson `r` is dominated by fist/peace. Consider running regression on those two classes only for an honest decoding number.
- **Thumb decreases during `fist` (0.6 → 0.27).** Anatomically expected — the thumb wraps over the curled fingers rather than flexing inward — but it means we can't summarize "grip" by averaging across fingers. The Ridge regression in `08` predicts each finger independently, so it handles this correctly.
- **Trial-averaged patterns are highly stereotyped.** Sharp transitions, consistent plateau heights across all 30 trials per class. The participant performed each gesture nearly identically every time, which means the ECoG correlates have high SNR after trial averaging. The bottleneck on decoding accuracy is sample count (30/class), not behavioral noise.
- **Resting baseline isn't fully relaxed** (thumb ~0.6, others ~0.15). If we ever want per-trial baselining for the regression, this is where it would help.